# 🔬 Notebook 07: External Validation (Messidor-2)
## Evaluate Model Generalization on External Dataset

**Objectives:**
1. Load trained model from APTOS 2019
2. Evaluate on Messidor-2 dataset (external validation)
3. Calculate generalization gap
4. Identify domain shift issues

**Why External Validation?**
- APTOS test set might have similar distribution to training
- Real clinical deployment uses different cameras/populations
- Messidor-2 is a standard benchmark (1,748 images)
- Generalization gap < 5% indicates robust model

**Note:** This notebook requires Messidor-2 images to be downloaded separately.
Download from: https://www.adcis.net/en/third-party/messidor2/

---

## 1. Setup & Imports

In [ ]:
# Core imports
import os
import sys
from pathlib import Path

# Deep Learning
import torch
import torch.nn.functional as F

# Data
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Metrics
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    cohen_kappa_score
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Progress
from tqdm.notebook import tqdm

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"✅ Using device: {device}")

In [ ]:
# Define paths
PROJECT_ROOT = Path("/Users/shivasaivemula/ALP Project/deep-retina-grade")
MESSIDOR_DIR = PROJECT_ROOT / "messidor_data"  # Put Messidor-2 images here
MESSIDOR_CSV = PROJECT_ROOT / "archive" / "messidor_data.csv"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
SRC_DIR = PROJECT_ROOT / "src"

sys.path.insert(0, str(SRC_DIR))

# ImageNet normalization
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

## 2. Check Data Availability

In [ ]:
# Check if Messidor data exists
print("📂 Checking Messidor-2 data availability...\n")

# Check CSV
if MESSIDOR_CSV.exists():
    df_messidor = pd.read_csv(MESSIDOR_CSV)
    print(f"✅ Messidor CSV found: {len(df_messidor)} records")
    print(f"   Columns: {df_messidor.columns.tolist()}")
    print(f"\n   Grade distribution:")
    print(df_messidor['adjudicated_dr_grade'].value_counts().sort_index())
else:
    print(f"❌ Messidor CSV not found at: {MESSIDOR_CSV}")

# Check images
if MESSIDOR_DIR.exists():
    images = list(MESSIDOR_DIR.glob("*.tif")) + list(MESSIDOR_DIR.glob("*.jpg"))
    print(f"\n✅ Messidor images found: {len(images)} files")
else:
    print(f"\n⚠️ Messidor images directory not found: {MESSIDOR_DIR}")
    print("\n📥 Please download Messidor-2 images from:")
    print("   https://www.adcis.net/en/third-party/messidor2/")
    print(f"\n   Extract images to: {MESSIDOR_DIR}")

## 3. Load Model

In [ ]:
from models.efficientnet import RetinaModel
from preprocessing.preprocess import RetinaPreprocessor

# Load model - Using EfficientNet-B0
model = RetinaModel(num_classes=5, pretrained=False, backbone='efficientnet_b0')
checkpoint = torch.load(MODELS_DIR / 'efficientnet_b0_best.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

# Initialize preprocessor
preprocessor = RetinaPreprocessor(img_size=224)

print(f"✅ Model loaded (EfficientNet-B0)")
print(f"   Best APTOS Kappa: {checkpoint.get('best_kappa', 'N/A')}")

## 4. Create Messidor Dataset

In [ ]:
class MessidorDataset(Dataset):
    """
    Messidor-2 Dataset for external validation.
    """
    
    def __init__(self, csv_path, images_dir, preprocessor, transform=None):
        self.df = pd.read_csv(csv_path)
        self.images_dir = Path(images_dir)
        self.preprocessor = preprocessor
        self.transform = transform
        
        # Filter to gradable images only
        self.df = self.df[self.df['adjudicated_gradable'] == 1].reset_index(drop=True)
        
        # Check which images exist
        self.valid_indices = []
        for idx, row in self.df.iterrows():
            img_path = self.images_dir / row['image_id']
            if img_path.exists():
                self.valid_indices.append(idx)
        
        print(f"Found {len(self.valid_indices)} / {len(self.df)} images")
    
    def __len__(self):
        return len(self.valid_indices)
    
    def __getitem__(self, idx):
        real_idx = self.valid_indices[idx]
        row = self.df.iloc[real_idx]
        
        # Load and preprocess image
        img_path = self.images_dir / row['image_id']
        img = self.preprocessor.preprocess(str(img_path))
        img_uint8 = (img * 255).astype(np.uint8)
        
        # Apply transforms
        if self.transform:
            transformed = self.transform(image=img_uint8)
            img_tensor = transformed['image']
        else:
            img_tensor = torch.from_numpy(img).permute(2, 0, 1).float()
        
        # Get label (DR grade)
        label = int(row['adjudicated_dr_grade'])
        
        return img_tensor, label

# Transform
val_transform = A.Compose([
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

print("✅ MessidorDataset class ready")

In [ ]:
# Create dataset if images exist
if MESSIDOR_DIR.exists() and any(MESSIDOR_DIR.glob("*")):
    messidor_dataset = MessidorDataset(
        csv_path=MESSIDOR_CSV,
        images_dir=MESSIDOR_DIR,
        preprocessor=preprocessor,
        transform=val_transform
    )
    
    messidor_loader = DataLoader(
        messidor_dataset,
        batch_size=16,
        shuffle=False,
        num_workers=0
    )
    
    print(f"\n✅ Messidor DataLoader created: {len(messidor_dataset)} images")
else:
    print("⚠️ Cannot create DataLoader - images not found")
    print("\nTo run external validation:")
    print("1. Download Messidor-2 from https://www.adcis.net/en/third-party/messidor2/")
    print(f"2. Extract images to: {MESSIDOR_DIR}")
    print("3. Re-run this notebook")

## 5. Evaluate on Messidor-2

In [ ]:
def evaluate_model(model, dataloader, device):
    """
    Evaluate model on a dataset.
    
    Returns:
        Dictionary with metrics
    """
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Evaluating"):
            images = images.to(device)
            
            outputs = model(images)
            probs = F.softmax(outputs, dim=1)
            preds = outputs.argmax(dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_preds)
    kappa = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
    cm = confusion_matrix(all_labels, all_preds)
    
    # Per-class metrics
    report = classification_report(all_labels, all_preds, output_dict=True, zero_division=0)
    
    # Severe DR sensitivity (Grade 3-4)
    severe_mask = all_labels >= 3
    if severe_mask.sum() > 0:
        severe_sensitivity = (all_preds[severe_mask] >= 3).sum() / severe_mask.sum()
    else:
        severe_sensitivity = 0.0
    
    return {
        'accuracy': accuracy,
        'quadratic_weighted_kappa': kappa,
        'severe_sensitivity': severe_sensitivity,
        'confusion_matrix': cm,
        'classification_report': report,
        'predictions': all_preds,
        'labels': all_labels
    }

print("✅ Evaluation function ready")

In [ ]:
# Run evaluation if dataset exists
if 'messidor_loader' in dir():
    print("🔬 Evaluating on Messidor-2...\n")
    messidor_results = evaluate_model(model, messidor_loader, device)
    
    print("=" * 60)
    print("📊 MESSIDOR-2 EXTERNAL VALIDATION RESULTS")
    print("=" * 60)
    print(f"Accuracy:                 {messidor_results['accuracy']:.4f}")
    print(f"Quadratic Weighted Kappa: {messidor_results['quadratic_weighted_kappa']:.4f}")
    print(f"Severe DR Sensitivity:    {messidor_results['severe_sensitivity']:.4f}")
    print("=" * 60)
else:
    print("⚠️ Messidor dataset not available - skipping evaluation")

## 6. Calculate Generalization Gap

In [ ]:
# Load APTOS test results for comparison
import json

aptos_metrics_path = RESULTS_DIR / "test_metrics.json"
if aptos_metrics_path.exists():
    with open(aptos_metrics_path) as f:
        aptos_results = json.load(f)
    
    print("📊 APTOS 2019 Test Set Results:")
    print(f"   Accuracy: {aptos_results['accuracy']:.4f}")
    print(f"   Kappa:    {aptos_results['quadratic_weighted_kappa']:.4f}")
    
    if 'messidor_results' in dir():
        print("\n" + "=" * 60)
        print("📈 GENERALIZATION GAP ANALYSIS")
        print("=" * 60)
        
        acc_gap = aptos_results['accuracy'] - messidor_results['accuracy']
        kappa_gap = aptos_results['quadratic_weighted_kappa'] - messidor_results['quadratic_weighted_kappa']
        
        print(f"\nAccuracy Gap:  {acc_gap:+.4f} ({'✅ Good' if abs(acc_gap) < 0.05 else '⚠️ High'})")
        print(f"Kappa Gap:     {kappa_gap:+.4f} ({'✅ Good' if abs(kappa_gap) < 0.05 else '⚠️ High'})")
        
        print("\n" + "-" * 60)
        print("Interpretation:")
        if abs(acc_gap) < 0.05 and abs(kappa_gap) < 0.05:
            print("✅ Model generalizes well to external data!")
            print("   Performance drop is within acceptable range (<5%).")
        else:
            print("⚠️ Significant generalization gap detected.")
            print("   Consider:")
            print("   - Domain adaptation techniques")
            print("   - More diverse training data")
            print("   - Different preprocessing for external cameras")
else:
    print("⚠️ APTOS test metrics not found")

## 7. Confusion Matrix Comparison

In [ ]:
if 'messidor_results' in dir():
    # Plot confusion matrix
    fig, ax = plt.subplots(figsize=(8, 6))
    
    cm = messidor_results['confusion_matrix']
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['No DR', 'Mild', 'Moderate', 'Severe', 'PDR'],
        yticklabels=['No DR', 'Mild', 'Moderate', 'Severe', 'PDR'],
        ax=ax
    )
    
    ax.set_title('Messidor-2 Confusion Matrix', fontsize=14, fontweight='bold')
    ax.set_xlabel('Predicted Grade')
    ax.set_ylabel('True Grade')
    
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'artifacts' / 'messidor_confusion_matrix.png', dpi=150)
    plt.show()
    
    print("\n✅ Saved: artifacts/messidor_confusion_matrix.png")
else:
    print("⚠️ No Messidor results to plot")

## 8. Save Results

In [ ]:
if 'messidor_results' in dir():
    # Save results
    results_to_save = {
        'dataset': 'Messidor-2',
        'n_samples': len(messidor_dataset),
        'accuracy': messidor_results['accuracy'],
        'quadratic_weighted_kappa': messidor_results['quadratic_weighted_kappa'],
        'severe_sensitivity': messidor_results['severe_sensitivity'],
        'confusion_matrix': messidor_results['confusion_matrix'].tolist(),
        'generalization_gap': {
            'accuracy': float(acc_gap) if 'acc_gap' in dir() else None,
            'kappa': float(kappa_gap) if 'kappa_gap' in dir() else None
        }
    }
    
    with open(RESULTS_DIR / 'messidor_validation.json', 'w') as f:
        json.dump(results_to_save, f, indent=2)
    
    print("✅ Saved: results/messidor_validation.json")

## 9. Summary

In [ ]:
print("=" * 70)
print("📊 EXTERNAL VALIDATION COMPLETE - SUMMARY")
print("=" * 70)

if 'messidor_results' in dir():
    print(f"""
✅ Messidor-2 Evaluation:
   • Samples:    {len(messidor_dataset)}
   • Accuracy:   {messidor_results['accuracy']:.4f}
   • Kappa:      {messidor_results['quadratic_weighted_kappa']:.4f}
   • Severe DR:  {messidor_results['severe_sensitivity']:.4f}

📈 Generalization Gap:
   • Accuracy:   {acc_gap:+.4f} ({'✅ <5%' if abs(acc_gap) < 0.05 else '⚠️ >5%'})
   • Kappa:      {kappa_gap:+.4f} ({'✅ <5%' if abs(kappa_gap) < 0.05 else '⚠️ >5%'})

📁 Artifacts:
   • results/messidor_validation.json
   • artifacts/messidor_confusion_matrix.png
""")
else:
    print("""
⚠️ External validation not completed.

To run external validation:

1. Download Messidor-2 dataset:
   https://www.adcis.net/en/third-party/messidor2/

2. Extract images to:
   deep-retina-grade/messidor_data/

3. Re-run this notebook

Note: External validation is optional but recommended for
demonstrating model robustness to your professors.
""")

print("=" * 70)